<a href="https://colab.research.google.com/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 2: Bayesian Probability

In this section we apply **Bayes' Theorem** to estimate the probability that a movie review is **positive**, given that a specific keyword appears in it.

We use the [IMDb Movie Reviews Dataset](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews) — 50,000 labelled reviews split evenly between `positive` and `negative` sentiment.

---

### Bayes' Theorem

$$P(\text{Positive} \mid \text{keyword}) = \frac{P(\text{keyword} \mid \text{Positive}) \cdot P(\text{Positive})}{P(\text{keyword})}$$

| Term | Name | What it means |
|---|---|---|
| $P(\text{Positive})$ | **Prior** | Overall fraction of positive reviews — our belief *before* seeing any keyword |
| $P(\text{keyword} \mid \text{Positive})$ | **Likelihood** | How often the keyword appears *inside* positive reviews |
| $P(\text{keyword})$ | **Marginal** | How often the keyword appears across *all* reviews |
| $P(\text{Positive} \mid \text{keyword})$ | **Posterior** | Updated probability that a review is positive, *given* the keyword is present |

## Step 1 — Import Libraries & Load Dataset

**What this cell does:**  
We import the four libraries we need — `pandas` to load and work with the dataset, `re` for regex keyword matching, `string` for punctuation removal, and `matplotlib` for visualisation. We then load the IMDb CSV file directly from a public GitHub URL into a DataFrame.

> **Note:** No machine learning libraries are used anywhere in this notebook. All probability calculations are done using basic Python operations only.

In [ ]:
import pandas as pd
import re
import string
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Load the IMDb dataset from a public GitHub mirror
DATASET_URL = (
    "https://raw.githubusercontent.com/kolaveridi/"
    "kaggle-imdb-sentiment-analysis-dataset/master/IMDB%20Dataset.csv"
)

df = pd.read_csv(DATASET_URL)

print(f"Dataset loaded successfully — {len(df):,} reviews")
print(f"Columns : {df.columns.tolist()}")
print()
print(df['sentiment'].value_counts().to_string())
print()
df.head(3)

## Step 2 — Keyword Selection

**What this cell does:**  
This is a planning step — no code is needed here. We choose 4 positive and 4 negative keywords based on words commonly found in movie reviews. The goal is to pick words that are *distinctive* — words that appear much more frequently in one sentiment class than the other, so the Bayesian update is meaningful.

| Category | Keywords | Reasoning |
|---|---|---|
| **Positive** | `brilliant`, `masterpiece`, `enjoyable`, `outstanding` | Strong praise words — rarely used in a negative context |
| **Negative** | `terrible`, `awful`, `boring`, `waste` | Strong criticism words — *waste* in particular signals "waste of time" |

We will compute **P(Positive | keyword)** for every keyword — i.e. how likely a review is positive *given* that word appears in it.

## Step 3 — Preprocessing

**What this cell does:**  
Before we count keyword occurrences we clean the review text in two ways:
1. **Lowercase** — converts `"Brilliant"` and `"BRILLIANT"` to `"brilliant"` so all matches are consistent regardless of how the reviewer typed the word.
2. **Remove punctuation** — strips characters like `!`, `.`, `,` so `"brilliant!"` still counts as a match for `brilliant`. We use `str.maketrans` which is a standard Python built-in — no libraries needed.

We store the cleaned text in a new column `review_clean` so the original `review` column is preserved.

In [ ]:
# Step 1: Convert all review text to lowercase
df['review_clean'] = df['review'].str.lower()

# Step 2: Remove all punctuation characters using a translation table
# str.maketrans creates a mapping that replaces every punctuation character with nothing
punct_table = str.maketrans('', '', string.punctuation)
df['review_clean'] = df['review_clean'].apply(
    lambda text: text.translate(punct_table)
)

print("Preprocessing complete.")
print()
print("Original  :", df['review'].iloc[1][:90], "...")
print("Cleaned   :", df['review_clean'].iloc[1][:90], "...")

## Step 4 — Calculate the Prior: P(Positive)

**What this cell does:**  
The **prior** is our starting belief — the probability that any randomly chosen review is positive, *before* we look at any keywords. We calculate it simply by counting positive reviews and dividing by the total.

$$P(\text{Positive}) = \frac{\text{Number of positive reviews}}{\text{Total reviews}}$$

Because the IMDb dataset is perfectly balanced (25,000 positive, 25,000 negative), we expect to get exactly 0.5. This means without any keyword evidence, a review has a 50/50 chance of being positive.

In [ ]:
# Count total and positive reviews
total_reviews  = len(df)
total_positive = len(df[df['sentiment'] == 'positive'])
total_negative = total_reviews - total_positive

# Prior probability: P(Positive) = positive reviews / total reviews
p_positive = total_positive / total_reviews

print(f"Total reviews    : {total_reviews:,}")
print(f"Positive reviews : {total_positive:,}")
print(f"Negative reviews : {total_negative:,}")
print()
print(f"P(Positive) = {total_positive} / {total_reviews} = {p_positive:.4f}")

## Step 5 — Implement Bayes' Theorem Functions

**What this cell does:**  
We define three small, focused functions — one for each probability component in Bayes' Theorem. Keeping them separate makes the code modular and easy to reuse (DRY principle).

- **`review_contains(text, keyword)`** — checks whether a keyword appears as a whole word in a review using regex. The `\b` (word boundary) anchor is important: without it, searching for `awful` would also match `awfully`.

- **`calc_p_keyword(keyword, dataframe)`** — calculates the **marginal** P(keyword): how common the keyword is across the *entire* dataset. This acts as a normalising constant in Bayes' Theorem.

- **`calc_p_keyword_given_positive(keyword, df_positive)`** — calculates the **likelihood** P(keyword | Positive): how often the keyword appears specifically within positive reviews.

- **`calc_posterior(likelihood, prior, marginal)`** — applies Bayes' Theorem directly to get P(Positive | keyword). Includes a guard against division by zero if a keyword never appears.

In [ ]:
# ── Helper: whole-word keyword check ──────────────────────────────────────────
def review_contains(text, keyword):
    """
    Returns True if `keyword` appears as a whole word in `text`.
    \\b is a word boundary — prevents 'bore' matching inside 'boring'.
    """
    pattern = r'\b' + re.escape(keyword) + r'\b'
    return bool(re.search(pattern, text))


# ── Marginal: P(keyword) ───────────────────────────────────────────────────────
def calc_p_keyword(keyword, dataframe):
    """
    P(keyword) = reviews containing keyword / total reviews
    Measures how common the keyword is across the ENTIRE dataset.
    """
    count = dataframe['review_clean'].apply(
        lambda t: review_contains(t, keyword)
    ).sum()
    return count / len(dataframe), int(count)


# ── Likelihood: P(keyword | Positive) ─────────────────────────────────────────
def calc_p_keyword_given_positive(keyword, df_positive):
    """
    P(keyword | Positive) = positive reviews containing keyword / total positive reviews
    Measures how often the keyword appears specifically within positive reviews.
    """
    count = df_positive['review_clean'].apply(
        lambda t: review_contains(t, keyword)
    ).sum()
    return count / len(df_positive), int(count)


# ── Posterior: P(Positive | keyword) via Bayes' Theorem ───────────────────────
def calc_posterior(likelihood, prior, marginal):
    """
    Applies Bayes' Theorem:
        P(Positive | keyword) = P(keyword | Positive) * P(Positive) / P(keyword)
    Returns 0 if the keyword never appears (avoids division by zero).
    """
    if marginal == 0:
        return 0.0
    return (likelihood * prior) / marginal


# ── Create the positive-only subset for use in likelihood calculations ─────────
df_positive = df[df['sentiment'] == 'positive'].copy()

# Quick sanity check on one keyword
print("Functions defined. Quick test on 'brilliant':")
_marg, _mc = calc_p_keyword('brilliant', df)
_like, _lc = calc_p_keyword_given_positive('brilliant', df_positive)
_post      = calc_posterior(_like, p_positive, _marg)
print(f"  P('brilliant')           = {_marg:.4f}  ({_mc} reviews)")
print(f"  P('brilliant'|Positive)  = {_like:.4f}  ({_lc} positive reviews)")
print(f"  P(Positive|'brilliant')  = {_post:.4f}")

## Step 6 — Compute All Probabilities

**What this cell does:**  
We loop through all 8 keywords and call our three functions for each one. For every keyword we compute and store:
- The **prior** (same for all keywords since it's a dataset-level property)
- The **likelihood** P(keyword | Positive)
- The **marginal** P(keyword)
- The **posterior** P(Positive | keyword)

All results are stored in a list of dictionaries so we can display and plot them in the next steps.

In [ ]:
# Define our keyword lists
POSITIVE_KEYWORDS = ['brilliant', 'masterpiece', 'enjoyable', 'outstanding']
NEGATIVE_KEYWORDS = ['terrible',  'awful',       'boring',    'waste']
ALL_KEYWORDS      = POSITIVE_KEYWORDS + NEGATIVE_KEYWORDS

# Loop through every keyword and compute the four probability components
results = []

for kw in ALL_KEYWORDS:
    category            = 'Positive' if kw in POSITIVE_KEYWORDS else 'Negative'
    marginal,  cnt_all  = calc_p_keyword(kw, df)
    likelihood, cnt_pos = calc_p_keyword_given_positive(kw, df_positive)
    posterior           = calc_posterior(likelihood, p_positive, marginal)

    results.append({
        'keyword'   : kw,
        'category'  : category,
        'prior'     : round(p_positive,  4),
        'likelihood': round(likelihood,  4),
        'marginal'  : round(marginal,    4),
        'posterior' : round(posterior,   4),
        'cnt_all'   : cnt_all,
        'cnt_pos'   : cnt_pos,
    })
    print(f"  '{kw}' [{category:8s}] — total: {cnt_all:5d} reviews  |  in positive: {cnt_pos:5d}")

print("\nAll keywords processed.")

## Step 7 — Results Table

**What this cell does:**  
We print a formatted table showing all four Bayesian probability components for every keyword side by side. This makes it easy to compare how each word affects the posterior and to spot the pattern — positive keywords push the posterior above 0.5, negative keywords push it below.

In [ ]:
# Build and print a formatted results table
col_w   = [14, 10, 13, 13, 11, 11]
headers = ['Keyword', 'Category', 'P(Positive)', 'P(kw|Pos)', 'P(kw)', 'P(Pos|kw)']

header_row  = "| " + " | ".join(f"{h:<{w}}" for h, w in zip(headers, col_w)) + " |"
divider_row = "|" + "|".join("-" * (w + 2) for w in col_w) + "|"

print(header_row)
print(divider_row)

for r in results:
    vals = [
        r['keyword'], r['category'],
        f"{r['prior']:.4f}",
        f"{r['likelihood']:.4f}",
        f"{r['marginal']:.4f}",
        f"{r['posterior']:.4f}"
    ]
    print("| " + " | ".join(f"{v:<{w}}" for v, w in zip(vals, col_w)) + " |")

## Step 8 — Visualisation

**What this cell does:**  
We create two side-by-side bar charts:

- **Plot 1 (Posterior per keyword)** — shows P(Positive | keyword) for all 8 keywords. Green bars are positive keywords, red bars are negative. The dashed grey line at 0.5 is the prior baseline. Keywords above the line increase our belief the review is positive; keywords below decrease it.

- **Plot 2 (Likelihood vs Marginal)** — shows *why* the posterior shifts. When the blue bar (likelihood) is taller than the grey bar (marginal), the keyword appears more often in positive reviews than in the overall dataset — a positive signal. When the blue bar is shorter, it's a negative signal.

In [ ]:
keywords    = [r['keyword']    for r in results]
posteriors  = [r['posterior']  for r in results]
likelihoods = [r['likelihood'] for r in results]
marginals   = [r['marginal']   for r in results]
colours     = ['#27ae60' if r['category'] == 'Positive' else '#e74c3c' for r in results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Bayesian Sentiment Analysis — IMDb Movie Reviews', fontsize=14, fontweight='bold')

# ── Plot 1: Posterior probabilities ───────────────────────────────────────────
bars = ax1.bar(keywords, posteriors, color=colours, edgecolor='black', linewidth=0.7)
for bar, val in zip(bars, posteriors):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.012,
        f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold'
    )
ax1.axhline(0.5, color='grey', linestyle='--', linewidth=1.2)
ax1.set_ylim(0, 1.1)
ax1.set_ylabel('P(Positive | keyword)', fontsize=11)
ax1.set_title('Posterior Probability per Keyword', fontsize=12)
ax1.tick_params(axis='x', rotation=30)
ax1.grid(axis='y', alpha=0.3)
green_p = mpatches.Patch(color='#27ae60', label='Positive keyword')
red_p   = mpatches.Patch(color='#e74c3c', label='Negative keyword')
base_l  = plt.Line2D([0], [0], color='grey', linestyle='--', label='Prior baseline (0.5)')
ax1.legend(handles=[green_p, red_p, base_l], fontsize=9)

# ── Plot 2: Likelihood vs Marginal ────────────────────────────────────────────
x_pos = range(len(keywords))
width = 0.35
ax2.bar([p - width/2 for p in x_pos], likelihoods, width,
        color='#3498db', edgecolor='black', linewidth=0.7, label='P(kw | Positive) — Likelihood')
ax2.bar([p + width/2 for p in x_pos], marginals, width,
        color='#95a5a6', edgecolor='black', linewidth=0.7, label='P(kw) — Marginal')
ax2.set_xticks(list(x_pos))
ax2.set_xticklabels(keywords, rotation=30, ha='right')
ax2.set_ylabel('Probability', fontsize=11)
ax2.set_title('Likelihood vs Marginal per Keyword', fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Step 9 — Per-Keyword Calculation Breakdown

**What this cell does:**  
For each keyword we print a formatted summary block showing every step of the Bayes' Theorem calculation explicitly — including the intermediate numerator before dividing by the marginal. This makes the arithmetic fully transparent and easy to follow or verify by hand.

In [ ]:
def print_keyword_summary(r):
    """Prints a full Bayes' Theorem breakdown for one keyword."""
    sep = '─' * 54
    print(sep)
    print(f"  Keyword : '{r['keyword']}'  [{r['category']} keyword]")
    print(sep)
    print(f"  {'Component':<34} {'Value':>10}")
    print(f"  {'-'*34} {'-'*10}")
    print(f"  {'Prior       P(Positive)':<34} {r['prior']:>10.4f}")
    print(f"  {'Likelihood  P(kw | Positive)':<34} {r['likelihood']:>10.4f}")
    print(f"  {'Marginal    P(kw)':<34} {r['marginal']:>10.4f}")
    print(f"  {'Posterior   P(Positive | kw)':<34} {r['posterior']:>10.4f}")
    print()
    numerator = round(r['likelihood'] * r['prior'], 6)
    print(f"  Calculation (Bayes' Theorem):")
    print(f"    P(Pos|kw)  =  P(kw|Pos)  x  P(Pos)  /  P(kw)")
    print(f"               =  {r['likelihood']:.4f}  x  {r['prior']:.4f}  /  {r['marginal']:.4f}")
    print(f"               =  {numerator:.6f}  /  {r['marginal']:.4f}")
    print(f"               =  {r['posterior']:.4f}")
    print(sep)
    print()

# Print summary for every keyword
for r in results:
    print_keyword_summary(r)

## Step 10 — Discussion & Interpretation

**What this cell does:**  
This is a markdown cell summarising what the results mean. No code — just explanation.

---

### What the prior tells us
The IMDb dataset is perfectly balanced: 25,000 positive and 25,000 negative reviews, giving a prior $P(\text{Positive}) = 0.5$. Without any keyword evidence, a randomly selected review has a 50% chance of being positive. This is our starting point for every keyword.

### How positive keywords shift the posterior
Words like `masterpiece`, `brilliant`, and `outstanding` all produced posteriors **well above 0.5**. This happens because their **likelihood** $P(\text{kw} \mid \text{Positive})$ is much higher than their **marginal** $P(\text{kw})$ — they appear disproportionately often in positive reviews. Bayes' Theorem captures this: when a keyword is more likely in a positive review than in a random review, our belief that the review is positive increases.

### How negative keywords shift the posterior
Words like `terrible`, `awful`, and `boring` produced posteriors **well below 0.5**. Their likelihood inside positive reviews is very low — these words are rarely used to praise a film. Even though they appear a reasonable number of times across all reviews, their near-absence in positive reviews causes the posterior to drop sharply.

### The role of the marginal (Plot 2)
The marginal $P(\text{keyword})$ is a normalising constant — it prevents over-counting by adjusting for how common the word is across the whole dataset. A rarer but more decisive word like `masterpiece` will produce a stronger posterior shift than a common word like `boring` which appears in both positive and negative reviews.

### Key insight from Plot 2
Whenever the **blue bar** (likelihood) is **taller** than the **grey bar** (marginal), the keyword is a positive signal — it appears more in positive reviews than chance alone would predict. Whenever the **blue bar is shorter**, it is a negative signal.

### Limitations
This model treats each word independently and ignores context. A phrase like *"not brilliant"* would still match `brilliant` and incorrectly push the posterior upward. More advanced approaches such as negation handling, n-grams, or a full Naive Bayes classifier would address this — but those fall outside the scope of this assignment.